# Trabajo Práctico Integrador: Análisis de Churn en Telecomunicaciones (Telco)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LauritoFarias/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)

**Materia:** Análisis de Datos
**Dataset:** Telco Customer Churn (Kaggle)

# A. Fundamentos, Ingesta y Limpieza de Datos

## 1. Configuración e Ingesta de Datos
* **Objetivo de negocio:** Identificar patrones de cancelación (*Churn*) en clientes de telecomunicaciones para reducir la tasa de baja.
* **Variables objetivo:** `Churn` (clasificación/fuga) y `TotalCharges` (regresión).
* **Hipótesis principales:** Mayor propensión de baja en contratos mensuales ($H_1$), en los primeros 6 meses de antigüedad ($H_2$) y en planes de Fibra Óptica con cargos elevados sin servicios de soporte ($H_3$). *(Ver desarrollo completo en `docs/INFORME_EJECUTIVO.pdf`).*

A continuación, se importan las bibliotecas y se realiza la ingesta directa del dataset estructurado (`.csv`).

In [ ]:
# =============================================================================
# 1. CONFIGURACIÓN DEL ENTORNO E INGESTA DE DATOS
# =============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid", palette="muted")

RUTA_LOCAL = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

URL_GITHUB = "https://raw.githubusercontent.com/LauritoFarias/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if os.path.exists(RUTA_LOCAL):
    print(f"Cargando dataset desde entorno local: {RUTA_LOCAL}")
    df_raw = pd.read_csv(RUTA_LOCAL)
else:
    print("Entorno en la nube detectado (Colab). Cargando dataset desde GitHub...")
    df_raw = pd.read_csv(URL_GITHUB)

df = df_raw.copy()

print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head(3))

## 2. Relevamiento de Variables vs. Tipos en Pandas (`dtypes`)
Se genera una tabla de auditoría para contrastar la clasificación estadística teórica de los 21 atributos frente a los tipos de datos asignados por defecto en Pandas (`dtypes`), identificando inconsistencias de tipado previo a la limpieza.

In [ ]:
# =============================================================================
# 2. RELEVAMIENTO DE VARIABLES: CLASIFICACIÓN TEÓRICA VS PANDAS DTYPES
# =============================================================================

clasificacion_teorica = {
    'customerID': ('Cualitativa Nominal (Identificador)', 'Irrelevante para modelado'),
    'gender': ('Cualitativa Nominal (Dicotómica)', 'Explicativa'),
    'SeniorCitizen': ('Cualitativa Nominal (Dicotómica: 0/1)', 'Explicativa'),
    'Partner': ('Cualitativa Nominal (Dicotómica)', 'Explicativa'),
    'Dependents': ('Cualitativa Nominal (Dicotómica)', 'Explicativa'),
    'tenure': ('Cuantitativa Discreta (Meses)', 'Explicativa'),
    'PhoneService': ('Cualitativa Nominal (Dicotómica)', 'Explicativa'),
    'MultipleLines': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'InternetService': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'OnlineSecurity': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'OnlineBackup': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'DeviceProtection': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'TechSupport': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'StreamingTV': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'StreamingMovies': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'Contract': ('Cualitativa Ordinal (Mensual < 1 Año < 2 Años)', 'Explicativa'),
    'PaperlessBilling': ('Cualitativa Nominal (Dicotómica)', 'Explicativa'),
    'PaymentMethod': ('Cualitativa Nominal (Politómica)', 'Explicativa'),
    'MonthlyCharges': ('Cuantitativa Continua (Monetaria)', 'Explicativa'),
    'TotalCharges': ('Cuantitativa Continua (Monetaria)', 'Explicativa'),
    'Churn': ('Cualitativa Nominal (Dicotómica)', 'Variable Objetivo (Target)')
}

relevamiento_df = pd.DataFrame({
    'Columna': df.columns,
    'Tipo en Pandas (dtype)': df.dtypes.values,
    'Valores Nulos (NaN)': df.isnull().sum().values,
    'Valores Únicos': [df[col].nunique() for col in df.columns],
    'Clasificación Estadística Teórica': [clasificacion_teorica[col][0] for col in df.columns],
    'Rol en el Análisis': [clasificacion_teorica[col][1] for col in df.columns]
})

display(relevamiento_df)

## 3. Limpieza de Datos y Adecuación de Tipos
En este bloque se sanea la calidad del dataset ejecutando tres acciones:
1. **Detección de cadenas anómalas:** Se identifican 11 registros con espacios vacíos (`" "`) en `TotalCharges`.
2. **Imputación fundamentada:** Al verificarse que corresponden a clientes nuevos con antigüedad `tenure == 0`, se convierten a numérico (`float64`) e imputan con `0.0`.
3. **Conversión de `dtypes`:** Se recodifica `SeniorCitizen` a `Yes`/`No` y se transforman las variables cualitativas al tipo `category` (definiendo jerarquía ordinal para `Contract`).

In [ ]:
# =============================================================================
# 3. LIMPIEZA DE DATOS, CADENAS ANÓMALAS Y ADECUACIÓN DE DTYPES
# =============================================================================

print("--- Auditoría de espacios en blanco (' ') ---")
for col in df.select_dtypes(include='object').columns:
    espacios = (df[col].str.strip() == "").sum()
    if espacios > 0:
        print(f"Columna '{col}': {espacios} registros con cadenas vacías.")

casos_anomalos = df[df['TotalCharges'].str.strip() == ""]
print("\nVerificación de antigüedad (tenure) en registros con TotalCharges vacío:")
display(casos_anomalos[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'}).astype('category')

orden_contratos = pd.CategoricalDtype(categories=['Month-to-month', 'One year', 'Two year'], ordered=True)
df['Contract'] = df['Contract'].astype(orden_contratos)

cols_categoricas = [col for col in df.select_dtypes(include='object').columns if col != 'customerID']
for col in cols_categoricas:
    df[col] = df[col].astype('category')

print("\n--- Verificación de dtypes tras la adecuación ---")
print(df.dtypes)
print(f"\nTotal de valores nulos remanentes en el dataset: {df.isnull().sum().sum()}")

## 4. Detección de Valores Atípicos (Método IQR)
Se calculan los límites del Rango Intercuartílico ($Q_1 - 1.5 \times \text{IQR}$ y $Q_3 + 1.5 \times \text{IQR}$) y se grafican los *Boxplots* para las variables numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`). Al no registrarse valores fuera de los umbrales teóricos (0% outliers), se conserva la totalidad de los registros.

In [ ]:
# =============================================================================
# 4. DETECCIÓN ANALÍTICA Y VISUAL DE VALORES ATÍPICOS (MÉTODO IQR)
# =============================================================================

variables_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

resumen_iqr = []

for col in variables_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < limite_inferior) | (df[col] > limite_superior)]
    
    resumen_iqr.append({
        'Variable': col,
        'Q1 (25%)': Q1,
        'Mediana (50%)': df[col].median(),
        'Q3 (75%)': Q3,
        'IQR': IQR,
        'Límite Inferior': limite_inferior,
        'Límite Superior': limite_superior,
        'Cant. Outliers': len(outliers),
        '% Outliers': (len(outliers) / len(df)) * 100
    })

df_iqr = pd.DataFrame(resumen_iqr)
display(df_iqr)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(variables_numericas):
    sns.boxplot(x=df[col], ax=axes[i], color='steelblue', width=0.4)
    axes[i].set_title(f'Boxplot de {col}', fontweight='bold')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

## 5. Manipulación de Datos: Transformación, Filtrado y Agregación

Con el dataset limpio y tipificado, se ejecutan las operaciones requeridas de manipulación en Pandas:
* **Transformación:** Eliminación del identificador irrelevante (`customerID`), creación de columnas calculadas (`TheoreticalTotalCharges`, `BillingVariance`, `AvgHistoricalMonthlyCharge`) y demostración de renombrado de atributos.
* **Filtrado avanzado:** Aislamiento del segmento crítico de fuga combinando condiciones lógicas.
* **Agregación (`groupby`):** Resumen estadístico multidimensional (conteo, suma, media, mediana y tasa de *Churn*) segmentado por tipo de contrato y servicio de internet.

In [ ]:
# =============================================================================
# 4. MANIPULACIÓN DE DATOS: TRANSFORMACIÓN, FILTRADO Y AGREGACIÓN
# =============================================================================

if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

df['TheoreticalTotalCharges'] = df['tenure'] * df['MonthlyCharges']
df['BillingVariance'] = df['TotalCharges'] - df['TheoreticalTotalCharges']

df['AvgHistoricalMonthlyCharge'] = np.where(
    df['tenure'] > 0, 
    df['TotalCharges'] / df['tenure'], 
    df['MonthlyCharges']
)

df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

df.rename(columns={
    'tenure': 'TenureMonths',
    'SeniorCitizen': 'IsSeniorCitizen'
}, inplace=True)
display(df[['TenureMonths', 'MonthlyCharges', 'TotalCharges', 'TheoreticalTotalCharges', 'BillingVariance']].head())

filtro_alto_riesgo = (
    (df['Contract'] == 'Month-to-month') & 
    (df['InternetService'] == 'Fiber optic') & 
    ((df['MonthlyCharges'] > 70) | (df['TenureMonths'] < 6)) &
    (df['Churn'] == 'Yes')
)

df_segmento_critico = df[filtro_alto_riesgo]
print(f"Clientes en el segmento crítico filtrado: {len(df_segmento_critico)} ({len(df_segmento_critico)/len(df)*100:.2f}% del total)")
display(df_segmento_critico[['IsSeniorCitizen', 'TenureMonths', 'InternetService', 'Contract', 'MonthlyCharges', 'TotalCharges']].head())

resumen_negocio = df.groupby(['Contract', 'InternetService'], observed=False).agg(
    Total_Clientes=('Churn', 'count'),
    Clientes_Perdidos=('Churn_Binary', 'sum'),
    Tasa_Churn_Pct=('Churn_Binary', lambda x: round(x.mean() * 100, 2)),
    Antiguedad_Promedio_Meses=('TenureMonths', 'mean'),
    Cargo_Mensual_Promedio=('MonthlyCharges', 'mean'),
    Cargo_Mensual_Mediana=('MonthlyCharges', 'median'),
    Facturacion_Total_Acumulada=('TotalCharges', 'sum')
).reset_index()

print("\n--- Segmentación Estratégica por Contrato y Servicio de Internet ---")
display(resumen_negocio)